# 4 多模态任务：图理解、语音与图像生成

本章体验 LongCat-Next 的原生多模态能力：一次推理运行同时执行**图像理解**（book.png）、**语音理解**（math1.wav 转写）与**语音合成**（参考音色回答），再单独执行**图像生成**。视觉编码器、音频编码器、生成头与 vocoder/image_decoder 按需惰性加载，首次使用时从权重目录读入。


## 1 环境与工作区准备

复用第 2 章的统一环境与推理仓克隆（cann-recipes-infer 固定基线 + 单卡 patch）。多模态素材已通过软链 `requests/assets` 指向权重目录的官方 `assets/`（`book.png`、`math1.wav`、`system_audio.wav` 等）。


In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

print('[环境准备] 开始', flush=True)
os.environ.setdefault('ASCEND_RT_VISIBLE_DEVICES', '0')

def locate_repo_root():
    repo_roots = []
    for env_name in ('GITCODE_REPO_ROOT', 'CANN_LEARNING_HUB_ROOT', 'REPO_ROOT'):
        if os.environ.get(env_name):
            repo_roots.append(Path(os.environ[env_name]))
    try:
        cwd = Path.cwd()
        lineage = [cwd, *cwd.parents]
        repo_roots.extend(lineage)
        repo_roots.extend(base / 'cann-learning-hub' for base in lineage)
        for base in lineage:
            try:
                repo_roots.extend(base.glob('*/cann-learning-hub'))
            except OSError:
                pass
    except FileNotFoundError:
        pass
    repo_roots.append(Path('/opt/atomgit/cann-learning-hub'))

    seen = set()
    for repo_root in repo_roots:
        key = str(repo_root)
        if key in seen:
            continue
        seen.add(key)
        if (repo_root / 'tutorials/llm_inference/longcat_next/src').exists():
            return repo_root
    raise FileNotFoundError('Cannot locate cann-learning-hub repository root. Open this notebook inside cann-learning-hub or set CANN_LEARNING_HUB_ROOT to the repository root.')

REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT / 'tutorials' / 'llm_inference' / 'longcat_next'
SRC_DIR = TUTORIAL_DIR / 'src'
SOURCE_DIR = REPO_ROOT / 'Sources' / 'model_inference_optimization' / 'longcat_next'
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME is not set. Please activate the CANN environment before running this notebook.')
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env_cmd = f'source {cann_script} && env'
env = subprocess.check_output(f"bash -lc '{env_cmd}'", shell=True, text=True, cwd=REPO_ROOT)
for line in env.splitlines():
    if '=' not in line:
        continue
    key, value = line.split('=', 1)
    if key == 'PATH':
        # login shell 会触发 conda init 并激活 base，其 PATH 会让后续 %%bash 单元里的
        # python3 解析到 base 环境（本教程环境没有 diffusers 等依赖）。
        # 跳过 PATH，保住当前 kernel 解释器的优先级。
        continue
    os.environ.__setitem__(key, value)
# 把 CANN bin 追加到当前 kernel 的 PATH（不覆盖），infer.sh 内的 python3 仍解析到本环境
_kernel_bin = str(Path(sys.executable).parent)
_cann_bin = str(Path(os.environ.get('ASCEND_HOME_PATH', '/usr/local/Ascend/ascend-toolkit')) / 'bin')
os.environ['PATH'] = f'{_kernel_bin}:{_cann_bin}:{os.environ["PATH"]}'

from inference_scripts.recipe_workflow import (
    ensure_infer_repo,
    ensure_request_assets,
    prepare_runtime_yaml,
    print_yaml_focus,
    resolve_model_path,
)

# 复用（必要时自愈）第 2 章准备的克隆：固定基线 969160e + 单卡推理 patch
RECIPE_ROOT = ensure_infer_repo()
RECIPE_CONFIG_DIR = RECIPE_ROOT / 'models' / 'longcat_next' / 'config'
RECIPE_REQUESTS_DIR = RECIPE_ROOT / 'models' / 'longcat_next' / 'requests'
INFER_SH = RECIPE_ROOT / 'executor' / 'scripts' / 'infer.sh'
MODEL_PATH = resolve_model_path(quiet=True)
ensure_request_assets(MODEL_PATH, RECIPE_REQUESTS_DIR)

os.environ['LONGCAT_RECIPE_ROOT'] = str(RECIPE_ROOT)
os.environ['LONGCAT_MODEL_PATH'] = MODEL_PATH

print('infer 仓目录 =', RECIPE_ROOT)
print('模型权重 =', MODEL_PATH)
print('可见 NPU =', os.environ.get('ASCEND_RT_VISIBLE_DEVICES'))
print('[环境准备] 完成（依赖检查沿用第 2 章统一环境）', flush=True)



## 2 多模态请求清单

各任务的 `media` 键与产物（键名必须与任务要求**完全一致**，多写或少写都会报错；素材文件不存在直接报错）：

| task | media 键 | 产物 |
| --- | --- | --- |
| `text` | — | `text.txt` |
| `image_understanding` | `image` | `text.txt` |
| `image_generation` | — | `image*.png` |
| `audio_to_text` | `audio` | `text.txt` |
| `audio_to_audio` | `audio`、`reference_audio` | `audio*.wav` |
| `speech_synthesis` | `reference_audio` | `audio*.wav` |

本章使用的 `requests/multimodal.json` 包含三条请求：

1. **图像理解**：`What book is this?` + `assets/book.png`（使用官方示例风格的简短问句，避免 `enable_thinking=false` 下开放式中文长 prompt 的退化问题，见第 3 章提示）；
2. **语音理解**：转写 `assets/math1.wav`（一道小学数学题的朗读）；
3. **语音合成**：以 `assets/system_audio.wav` 为参考音色回答一段话，输出 24 kHz WAV。

图像理解会注入大量视觉 token（高分辨率图可达 2000+），因此本 YAML 的 `input_truncated_len`/`max_prefill_tokens` 扩到 **4096**——KV 容量不足会在 prefill 前直接报 `Sequence exceeds KV capacity`。


In [ ]:
from inference_scripts.recipe_workflow import read_request_manifest

MM_YAML_TEMPLATE = RECIPE_CONFIG_DIR / 'longcat_next_multimodal_single_npu_offload.yaml'
mm_requests = read_request_manifest(MM_YAML_TEMPLATE, recipe_root=RECIPE_ROOT)
print(json.dumps(mm_requests, ensure_ascii=False, indent=2))


## 3 执行多模态推理

三条请求按清单顺序串行执行。图理解约 1 分钟（视觉 token 的 prefill 占大头）、语音转写约 5 分钟（127 token 输出）、语音合成约 9 分钟（242 token，音频头逐步生成）；加上权重与组件加载，整轮约 15 分钟。


In [ ]:
import os
import time
from inference_scripts.recipe_workflow import (
    collect_request_metrics,
    prepare_runtime_yaml,
    print_yaml_focus,
)

MM_WORK_DIR = SOURCE_DIR / 'run_outputs' / 'multimodal'
# 运行 YAML 写入克隆仓 config/ 目录：上游 infer.sh 的 --yaml 只接受 config/ 下的文件名
MM_YAML = RECIPE_CONFIG_DIR / 'multimodal.yaml'
MM_WORK_DIR.mkdir(parents=True, exist_ok=True)

print('[多模态推理] 开始', flush=True)
prepare_runtime_yaml(MM_YAML_TEMPLATE, MM_YAML, model_path=MODEL_PATH)
print_yaml_focus(MM_YAML)

os.environ['LONGCAT_RUN_YAML'] = str(MM_YAML)
os.environ['LONGCAT_RUN_YAML_NAME'] = MM_YAML.name  # infer.sh --yaml 传文件名
os.environ['LONGCAT_RUN_LOG'] = str(MM_WORK_DIR / 'infer_stdout.log')
os.environ['LONGCAT_RUN_START_TIME'] = str(time.perf_counter())
print('运行 YAML =', MM_YAML)


In [ ]:
%%bash
set -euo pipefail

echo 'recipes 启动命令:'
echo "cd ${LONGCAT_RECIPE_ROOT}"
echo "bash executor/scripts/infer.sh --model longcat_next --yaml ${LONGCAT_RUN_YAML_NAME}"
cd "${LONGCAT_RECIPE_ROOT}"
bash executor/scripts/infer.sh --model longcat_next --yaml "${LONGCAT_RUN_YAML_NAME}" 2>&1 | tee "${LONGCAT_RUN_LOG}"


## 4 查看多模态产物

读取各请求的 `result.json`，并查看文本产物与合成音频的属性（24 kHz 单声道 WAV，由官方 cosy24k vocoder 输出）。语音合成任务是"**用参考音频的音色来回答**"，而非逐字 TTS——prompt 是模型要回答的内容，音色来自 `reference_audio`。


In [ ]:
mm_metrics = collect_request_metrics(
    RECIPE_ROOT,
    MM_WORK_DIR,
    'multimodal',
    MM_YAML,
)
print(f"{'请求':<28} {'task':<20} {'tokens':>7} {'耗时(s)':>9} {'产物'}")
for row in mm_metrics['requests']:
    files = row['image_files'] + row['audio_files'] or ['-']
    print(f"{row['id']:<28} {row['task']:<20} {row['generated_sequence_tokens']:>7} "
          f"{row['autoregressive_seconds']:>9.1f} {', '.join(files)}")

results_dir = Path(mm_metrics['recipe_case_dir']) / 'results'
print('\n[图像理解] 生成文本:')
print((results_dir / 'image_understanding_book' / 'text.txt').read_text(encoding='utf-8')[:600])
print('\n[语音理解] 转写文本:')
print((results_dir / 'audio_to_text_math' / 'text.txt').read_text(encoding='utf-8')[:600])

import soundfile as sf
audio_path = results_dir / 'speech_synthesis_intro' / 'audio_0.wav'
info = sf.info(str(audio_path))
print(f"\n[语音合成] audio_0.wav: {info.duration:.1f}s, {info.samplerate}Hz, "
      f"{info.channels}ch, subtype={info.subtype}")
print((results_dir / 'speech_synthesis_intro' / 'text.txt').read_text(encoding='utf-8'))


## 5 图像生成

图像生成任务把用户消息以 `<longcat_img_start>` 结尾触发：深度 Transformer 生成头按 `token_h × token_w` 网格产出离散视觉 code，再由 image_decoder + refiner 解码为 PNG。要点：

- 出现 `image_generation` 请求时，代码自动把 `scheduler_config.batch_size` 提到 **2** 以启用 CFG（无分类器引导），无需手工修改；
- 官方默认 `token_h=token_w=37`（1369 组 code，耗时较长）；本章用请求级覆盖 `visual_generation_config.custom_params` 调低为 **16×16**，兼顾演示时长，产物为 448×445 像素 PNG；
- `max_new_tokens` 需覆盖全部 code 生成步数（本例 4096）。


In [ ]:
import os
import time
from inference_scripts.recipe_workflow import (
    collect_request_metrics,
    read_request_manifest,
    prepare_runtime_yaml,
    print_yaml_focus,
)

IGEN_YAML_TEMPLATE = RECIPE_CONFIG_DIR / 'longcat_next_image_gen_single_npu_offload.yaml'
IGEN_WORK_DIR = SOURCE_DIR / 'run_outputs' / 'image_gen'
# 运行 YAML 写入克隆仓 config/ 目录：上游 infer.sh 的 --yaml 只接受 config/ 下的文件名
IGEN_YAML = RECIPE_CONFIG_DIR / 'image_gen.yaml'
IGEN_WORK_DIR.mkdir(parents=True, exist_ok=True)

print('[图像生成] 开始', flush=True)
print('请求清单:')
print(json.dumps(read_request_manifest(IGEN_YAML_TEMPLATE, recipe_root=RECIPE_ROOT), ensure_ascii=False, indent=2))
prepare_runtime_yaml(IGEN_YAML_TEMPLATE, IGEN_YAML, model_path=MODEL_PATH)
print_yaml_focus(IGEN_YAML)

os.environ['LONGCAT_RUN_YAML'] = str(IGEN_YAML)
os.environ['LONGCAT_RUN_YAML_NAME'] = IGEN_YAML.name  # infer.sh --yaml 传文件名
os.environ['LONGCAT_RUN_LOG'] = str(IGEN_WORK_DIR / 'infer_stdout.log')
os.environ['LONGCAT_RUN_START_TIME'] = str(time.perf_counter())
print('运行 YAML =', IGEN_YAML)


In [ ]:
%%bash
set -euo pipefail

echo 'recipes 启动命令:'
echo "cd ${LONGCAT_RECIPE_ROOT}"
echo "bash executor/scripts/infer.sh --model longcat_next --yaml ${LONGCAT_RUN_YAML_NAME}"
cd "${LONGCAT_RECIPE_ROOT}"
bash executor/scripts/infer.sh --model longcat_next --yaml "${LONGCAT_RUN_YAML_NAME}" 2>&1 | tee "${LONGCAT_RUN_LOG}"


In [ ]:
igen_metrics = collect_request_metrics(
    RECIPE_ROOT,
    IGEN_WORK_DIR,
    'image_gen',
    IGEN_YAML,
)
results_dir = Path(igen_metrics['recipe_case_dir']) / 'results'
from PIL import Image
for row in igen_metrics['requests']:
    print(f"{row['id']}: {row['generated_sequence_tokens']} code tokens, "
          f"{row['autoregressive_seconds']:.1f}s, 产物={row['image_files']}")
for image_path in sorted(results_dir.glob('image*/*.png')):
    with Image.open(image_path) as img:
        print(f"{image_path.name}: {img.size[0]}x{img.size[1]} {img.mode}")
        print('保存路径 =', image_path)


## 6 小结

- 多模态任务只需在请求清单里声明 `task` 与 `media`，素材相对清单目录解析；图像理解注意 KV 容量（4096）与 prompt 风格，图像生成注意 `token_h/token_w` 与 `max_new_tokens` 预算。
- 语音合成/复刻的音频头是逐步生成的离散 code（242 token ≈ 9 分钟），是本章最慢的任务；框架支持 `enable_profiler` 开关（产物在 `prof/prefill`、`prof/decode` 下），可供延伸分析设备算子与 CPU offload 的耗时占比、解释单卡吞吐瓶颈。
